<a href="https://colab.research.google.com/github/NqobileNgidi/MIT805-Assignment/blob/main/COSMOPEDIA_MIT805_Cleaning_EDA_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip uninstall -y dataproc-spark-connect
%pip install -q -U pyspark==3.5.0 huggingface_hub pyarrow pandas==2.2.3 matplotlib seaborn

## IMPORTS


In [2]:
import shutil
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import snapshot_download
from pyspark.sql import SparkSession, functions as F



## DATA LOCATION


In [3]:

DATA_ROOT = Path(r"/content/cosmopedia_web_v1")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("Data location:", DATA_ROOT)



Data location: /content/cosmopedia_web_v1


## DISK CHECK


In [4]:
total, used, free = shutil.disk_usage(str(DATA_ROOT))
print(f"Total: {total/(1024**3):.2f} GiB")
print(f"Used:  {used/(1024**3):.2f} GiB")
print(f"Free:  {free/(1024**3):.2f} GiB")

if free < 45 * (1024**3):
    raise RuntimeError(
        f"Only {free/(1024**3):.2f} GiB is free on {DATA_ROOT}. "
        "Choose a drive with substantially more free space."
    )

Total: 107.72 GiB
Used:  21.30 GiB
Free:  86.40 GiB


## DOWNLOAD ONLY web_samples_v1


In [5]:
snapshot_path = snapshot_download(
    repo_id="HuggingFaceTB/cosmopedia",
    repo_type="dataset",
    allow_patterns=["data/web_samples_v1/*.parquet"],
    local_dir=str(DATA_ROOT),
    max_workers=1,
)
print("Downloaded to:", snapshot_path)



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 139 files:   0%|          | 0/139 [00:00<?, ?it/s]

Downloaded to: /content/cosmopedia_web_v1


## VERIFY SIZE


In [6]:
PARQUET_DIR = DATA_ROOT / "data" / "web_samples_v1"
parquet_files = sorted(PARQUET_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(f"No Parquet files found in {PARQUET_DIR}")

working_bytes = sum(p.stat().st_size for p in parquet_files)
print("Parquet files:", len(parquet_files))
print(f"Working dataset size: {working_bytes/(1024**3):.2f} GiB")



Parquet files: 139
Working dataset size: 36.30 GiB


## START SPARK


In [7]:
spark = (
    SparkSession.builder
    .appName("Cosmopedia_MIT805")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.files.maxPartitionBytes", "128m")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)



Spark: 3.5.0


## LOAD DATA


In [8]:
DATA_PATH = str(DATA_ROOT / "data" / "web_samples_v1" / "*.parquet")

df = spark.read.parquet(DATA_PATH)

print("Columns:")
print(df.columns)

print("\nSchema:")
df.printSchema()


Columns:
['text_token_length', 'prompt', 'text', 'seed_data', 'format', 'audience']

Schema:
root
 |-- text_token_length: long (nullable = true)
 |-- prompt: string (nullable = true)
 |-- text: string (nullable = true)
 |-- seed_data: string (nullable = true)
 |-- format: string (nullable = true)
 |-- audience: string (nullable = true)



## RAW RECORD COUNT


In [9]:
raw_row_count = df.count()
print(f"Raw records: {raw_row_count:,}")



Raw records: 12,426,348


## MISSING VALUES


In [10]:
df.select([
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in df.columns
]).show(truncate=False)



+-----------------+------+----+---------+------+--------+
|text_token_length|prompt|text|seed_data|format|audience|
+-----------------+------+----+---------+------+--------+
|0                |0     |0   |0        |0     |0       |
+-----------------+------+----+---------+------+--------+



## DUPLICATES


In [11]:
if "text" in df.columns:
    distinct_text_count = df.select("text").distinct().count()
    print(f"Distinct documents: {distinct_text_count:,}")
    print(f"Exact duplicate documents: {raw_row_count-distinct_text_count:,}")



Distinct documents: 12,426,348
Exact duplicate documents: 0


In [12]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

## DESCRIPTIVE STATISTICS


In [ ]:
if "text_token_length" in df.columns:
    df.select("text_token_length").summary(
        "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
    ).show()



## CATEGORICAL EDA


In [ ]:
for column in ["seed_data", "format", "audience"]:
    if column in df.columns:
        print(f"\nTop values: {column}")
        df.groupBy(column).count().orderBy(F.desc("count")).limit(20).show(truncate=False)



## DERIVED VARIABLES


In [ ]:
if "text" in df.columns:
    df = (
        df.withColumn("text_char_length", F.length("text"))
          .withColumn(
              "word_count",
              F.size(F.split(F.trim("text"), r"\s+"))
          )
    )



## CLEANING


In [ ]:
clean_df = df

if "text" in clean_df.columns:
    clean_df = (
        clean_df
        .withColumn("text", F.trim("text"))
        .filter(F.col("text").isNotNull())
        .filter(F.length("text") > 0)
    )

if "text_token_length" in clean_df.columns:
    clean_df = clean_df.filter(
        F.col("text_token_length").isNotNull()
        & (F.col("text_token_length") > 0)
    )

if "text" in clean_df.columns:
    clean_df = clean_df.dropDuplicates(["text"])



## RAW VS CLEANED


In [ ]:
clean_row_count = clean_df.count()
removed_rows = raw_row_count - clean_row_count
removal_percentage = (
    removed_rows / raw_row_count * 100 if raw_row_count else 0
)

print(f"Raw records: {raw_row_count:,}")
print(f"Cleaned records: {clean_row_count:,}")
print(f"Removed: {removed_rows:,}")
print(f"Removal rate: {removal_percentage:.2f}%")



## SAVE CLEANED DATA


In [ ]:
# WARNING: this creates another large dataset.
CLEAN_ROOT = DATA_ROOT / "cleaned"
CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

(
    clean_df.write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(str(CLEAN_ROOT))
)
print("Cleaned data saved to:", CLEAN_ROOT)



## VERIFY CLEANED SIZE


In [ ]:
cleaned_files = sorted(CLEAN_ROOT.glob("*.parquet"))
cleaned_bytes = sum(p.stat().st_size for p in cleaned_files)
print(f"Cleaned dataset size: {cleaned_bytes/(1024**3):.2f} GiB")



## SAFE SAMPLE FOR PLOTS


In [ ]:
# NEVER call toPandas() on the complete 12GB+ dataset.
sample_pd = (
    clean_df
    .sample(withReplacement=False, fraction=0.01, seed=42)
    .limit(100_000)
    .toPandas()
)
print("Plot sample:", len(sample_pd))



## TOKEN LENGTH


In [ ]:
if "text_token_length" in sample_pd.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(sample_pd["text_token_length"].dropna(), bins=50)
    plt.title("Cosmopedia Text Token Length Distribution")
    plt.xlabel("Token length")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()



## WORD COUNT


In [ ]:
if "word_count" in sample_pd.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(sample_pd["word_count"].dropna(), bins=50)
    plt.title("Cosmopedia Word Count Distribution")
    plt.xlabel("Word count")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()



## SEED DATA


In [ ]:
if "seed_data" in sample_pd.columns:
    counts = sample_pd["seed_data"].value_counts().head(15).sort_values()
    plt.figure(figsize=(10, 6))
    counts.plot(kind="barh")
    plt.title("Top Cosmopedia Seed Data Sources")
    plt.xlabel("Sampled records")
    plt.ylabel("Seed data")
    plt.tight_layout()
    plt.show()



## FORMAT


In [ ]:
if "format" in sample_pd.columns:
    counts = sample_pd["format"].value_counts().head(15).sort_values()
    plt.figure(figsize=(10, 6))
    counts.plot(kind="barh")
    plt.title("Cosmopedia Content Format Distribution")
    plt.xlabel("Sampled records")
    plt.ylabel("Format")
    plt.tight_layout()
    plt.show()



## AUDIENCE


In [ ]:
if "audience" in sample_pd.columns:
    counts = sample_pd["audience"].value_counts().head(15).sort_values()
    plt.figure(figsize=(10, 6))
    counts.plot(kind="barh")
    plt.title("Cosmopedia Target Audience Distribution")
    plt.xlabel("Sampled records")
    plt.ylabel("Audience")
    plt.tight_layout()
    plt.show()



## EXPORT RESULTS


In [ ]:
RESULTS_ROOT = DATA_ROOT / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame({
    "metric": [
        "raw_records", "cleaned_records", "records_removed",
        "percentage_removed", "raw_dataset_size_GiB",
        "cleaned_dataset_size_GiB"
    ],
    "value": [
        raw_row_count, clean_row_count, removed_rows,
        round(removal_percentage, 4),
        round(working_bytes/(1024**3), 4),
        round(cleaned_bytes/(1024**3), 4)
    ]
})

summary.to_csv(RESULTS_ROOT / "cleaning_summary.csv", index=False)
sample_pd.to_csv(RESULTS_ROOT / "eda_sample.csv", index=False)

print(summary)
print("Results saved to:", RESULTS_ROOT)
